In [ ]:
import os
import cv2
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

In [ ]:
!pip install kaggle

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
!unzip "/content/drive/MyDrive/archive(1).zip" -d "/content/dataset"

Archive:  /content/drive/MyDrive/archive(1).zip
replace /content/dataset/final_kaggle_with_additional_video/barbell biceps curl/barbell biceps curl_1.mp4? [y]es, [n]o, [A]ll, [N]one, [r]ename: 

In [ ]:
dataset_path = "/content/dataset/final_kaggle_with_additional_video"

In [ ]:
class_names = sorted(os.listdir(dataset_path))
print(class_names)

NameError: name 'os' is not defined

In [ ]:
video_paths = []
labels = []

for class_name in class_names:
    class_folder = os.path.join(dataset_path, class_name)

    for file_name in os.listdir(class_folder):
        if file_name.endswith(".mp4"):
            video_paths.append(os.path.join(class_folder, file_name))
            labels.append(class_name)

print("Total videos:", len(video_paths))

Total videos: 101


In [ ]:
le = LabelEncoder()
encoded_labels = le.fit_transform(labels)

print("Classes:", le.classes_)

Classes: ['barbell biceps curl' 'hammer curl' 'push-up' 'shoulder press' 'squat']


In [ ]:
X_train_paths, X_temp_paths, y_train, y_temp = train_test_split(
    video_paths, encoded_labels, test_size=0.30, random_state=42, stratify=encoded_labels
)

X_val_paths, X_test_paths, y_val, y_test = train_test_split(
    X_temp_paths, y_temp, test_size=0.50, random_state=42, stratify=y_temp
)

print("Train videos:", len(X_train_paths))
print("Val videos:", len(X_val_paths))
print("Test videos:", len(X_test_paths))

Train videos: 70
Val videos: 15
Test videos: 16


In [ ]:
def load_video_clip(video_path, num_frames=16, target_size=(224, 224)):
    cap = cv2.VideoCapture(video_path)

    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    if total_frames <= 0:
        cap.release()
        raise ValueError(f"ভিডিও read করা যাচ্ছে না: {video_path}")

    # uniform frame indices
    indices = np.linspace(0, total_frames - 1, num_frames, dtype=int)

    frames = []
    frame_id = 0
    selected_set = set(indices.tolist())

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        if frame_id in selected_set:
            frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            frame = cv2.resize(frame, target_size)
            frame = frame.astype("float32") / 255.0
            frames.append(frame)

        frame_id += 1

    cap.release()

    # যদি কোনো কারণে frame কম পড়ে, pad করবো
    if len(frames) == 0:
        raise ValueError(f"কোনো frame পাওয়া যায়নি: {video_path}")

    while len(frames) < num_frames:
        frames.append(frames[-1])

    frames = np.array(frames[:num_frames], dtype=np.float32)

    return frames

In [ ]:
sample_clip = load_video_clip(X_train_paths[0], num_frames=16, target_size=(224, 224))
print(sample_clip.shape)

(16, 224, 224, 3)


In [ ]:
def build_dataset(video_path_list, label_list, num_frames=16, target_size=(224, 224)):
    clips = []
    clip_labels = []

    for video_path, label in zip(video_path_list, label_list):
        try:
            clip = load_video_clip(video_path, num_frames=num_frames, target_size=target_size)
            clips.append(clip)
            clip_labels.append(label)
        except Exception as e:
            print("Skipping:", video_path)
            print("Reason:", e)

    X = np.array(clips, dtype=np.float32)
    y = np.array(clip_labels, dtype=np.int32)

    return X, y

In [ ]:
X_train, y_train_final = build_dataset(X_train_paths, y_train, num_frames=16, target_size=(224, 224))
print(X_train.shape, y_train_final.shape)

NameError: name 'build_dataset' is not defined

In [ ]:
X_val, y_val_final = build_dataset(X_val_paths, y_val, num_frames=16, target_size=(224, 224))
print(X_val.shape, y_val_final.shape)

(15, 16, 224, 224, 3) (15,)


In [ ]:
X_test, y_test_final = build_dataset(X_test_paths, y_test, num_frames=16, target_size=(224, 224))
print(X_test.shape, y_test_final.shape)

(16, 16, 224, 224, 3) (16,)


In [ ]:
np.save("X_train_vivit.npy", X_train)
np.save("y_train_vivit.npy", y_train_final)

np.save("X_val_vivit.npy", X_val)
np.save("y_val_vivit.npy", y_val_final)

np.save("X_test_vivit.npy", X_test)
np.save("y_test_vivit.npy", y_test_final)

In [ ]:
import pickle

with open("label_encoder_vivit.pkl", "wb") as f:
    pickle.dump(le, f)

In [ ]:
X_train.shape = (70, 16, 224, 224, 3)
y_train.shape = (70,)

In [ ]:
print(np.unique(y_train_final))
print(np.unique(y_val_final))

[0 1 2 3 4]
[0 1 2 3 4]


In [ ]:
print(X_train.dtype)
print(y_train_final.dtype)

In [ ]:
num_frames = 16
target_size = (224, 224)

In [ ]:
sample_clip = load_video_clip(X_train_paths[0], num_frames=16, target_size=(224, 224))
print(sample_clip.shape)

(16, 224, 224, 3)


In [ ]:
sample_clip = load_video_clip(X_train_paths[0], num_frames=16, target_size=(224, 224))
print(sample_clip.shape)

(16, 224, 224, 3)


In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models

def create_vivit_model(
    num_frames=16,
    height=224,
    width=224,
    channels=3,
    num_classes=5
):
    inputs = layers.Input(shape=(num_frames, height, width, channels))

    # Spatial feature extraction for each frame
    x = layers.TimeDistributed(layers.Conv2D(32, (3, 3), activation="relu"))(inputs)
    x = layers.TimeDistributed(layers.MaxPooling2D((2, 2)))(x)

    x = layers.TimeDistributed(layers.Conv2D(64, (3, 3), activation="relu"))(x)
    x = layers.TimeDistributed(layers.MaxPooling2D((2, 2)))(x)

    x = layers.TimeDistributed(layers.Flatten())(x)

    # Temporal attention block
    attention_output = layers.MultiHeadAttention(num_heads=4, key_dim=64)(x, x)
    x = layers.Add()([x, attention_output])
    x = layers.LayerNormalization()(x)

    x_ffn = layers.Dense(128, activation="relu")(x)
    x_ffn = layers.Dense(x.shape[-1])(x_ffn)

    x = layers.Add()([x, x_ffn])
    x = layers.LayerNormalization()(x)

    x = layers.GlobalAveragePooling1D()(x)

    x = layers.Dense(128, activation="relu")(x)
    x = layers.Dropout(0.3)(x)

    outputs = layers.Dense(num_classes, activation="softmax")(x)

    model = models.Model(inputs, outputs)
    return model

In [ ]:
vivit_model = create_vivit_model(
    num_frames=16,
    height=224,
    width=224,
    channels=3,
    num_classes=5
)

vivit_model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 16, 224,   │          0 │ -                 │
│ (InputLayer)        │ 224, 3)           │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ time_distributed    │ (None, 16, 222,   │        896 │ input_layer[0][0] │
│ (TimeDistributed)   │ 222, 32)          │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ time_distributed_1  │ (None, 16, 111,   │          0 │ time_distributed… │
│ (TimeDistributed)   │ 111, 32)          │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ time_distributed_2  │ (None, 16, 109,   │     18,496 │ time_distributed… │
│ (TimeDistributed)   │ 109, 64)          │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ time_distributed_3  │ (None, 16, 54,    │          0 │ time_distributed… │
│ (TimeDistributed)   │ 54, 64)           │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ time_distributed_4  │ (None, 16,        │          0 │ time_distributed… │
│ (TimeDistributed)   │ 186624)           │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multi_head_attenti… │ (None, 16,        │ 191,290,3… │ time_distributed… │
│ (MultiHeadAttentio… │ 186624)           │            │ time_distributed… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add (Add)           │ (None, 16,        │          0 │ time_distributed… │
│                     │ 186624)           │            │ multi_head_atten… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalization │ (None, 16,        │    373,248 │ add[0][0]         │
│ (LayerNormalizatio… │ 186624)           │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 16, 128)   │ 23,888,000 │ layer_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, 16,        │ 24,074,496 │ dense[0][0]       │
│                     │ 186624)           │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_1 (Add)         │ (None, 16,        │          0 │ layer_normalizat… │
│                     │ 186624)           │            │ dense_1[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalizatio… │ (None, 16,        │    373,248 │ add_1[0][0]       │
│ (LayerNormalizatio… │ 186624)           │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_average_poo… │ (None, 186624)    │          0 │ layer_normalizat… │
│ (GlobalAveragePool… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_2 (Dense)     │ (None, 128)       │ 23,888,000 │ global_average_p… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_1 (Dropout) │ (None, 128)       │          0 │ dense_2[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_3 (Dense)     │ (None, 5)         │        645 │ dropout_1[0][0]   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 263,907,397 (1006.73 MB)

 Trainable params: 263,907,397 (1006.73 MB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
vivit_model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

In [ ]:
history_vivit = vivit_model.fit(
    X_train,
    y_train_final,
    validation_data=(X_val, y_val_final),
    epochs=20,
    batch_size=4
)

NameError: name 'X_train' is not defined